# Лекция: Компьютерное зрение
## Image Segmentation (part 2)

**Преподаватель:** Шевченко Елена, Т-Банк

**Аудитория:** МФТИ

---
### Оглавление
~~Part 1. Bacis: Задача, Метрики, Лоссы + Семантическая сегментация~~
- ~~1. Постановка задачи~~
- ~~2. Метрики~~
- ~~3. Функции потерь~~
- ~~4. Семантическая сегментация: U-Net, FPN, PAN, U-Net++, U²-Net, DeepLab v3, пример обучения~~


~~Part 2. Instance + Transformers~~
- ~~Instance segmentation: Mask R-CNN, Deep Watershed~~
- ~~Трансформеры в сегментации: SETR, SegFormer, Mask2Former~~


Part 3. SAM
- задача
- архитектура
- обучение, данные
- SAM 2 и SAM 3

# 7. Segment Anything Model (SAM, Kirillov et al., Meta AI, 2023)

![sam](assets/sam.jpg)

Авторы строят «foundation model» для сегментации из трёх согласованных частей:
**задача** (promptable segmentation) → **модель** (SAM) → **данные** (data engine и датасет SA-1B).

## 7.1 Задача: promptable segmentation

$$
M = f(I, \text{prompt})
$$

Модель получает изображение и **подсказку**, что сегментировать, и должна вернуть **валидную** маску — даже если подсказка неоднозначна.

| Prompt | Описание |
|------|------|
| Point | точка с меткой foreground / background (можно несколько) |
| Box | bounding box |
| Mask | грубая маска (например, предыдущее предсказание — для итеративного уточнения) |
| Text | только proof-of-concept в статье (см. 7.6); полноценно — в SAM 3 |

Ключевое отличие от моделей из `6-1`/`6-2`: SAM **не знает классов**. Он отвечает на вопрос «какой объект/часть здесь?», а не «что это?».
Благодаря этому — **zero-shot**: новые домены и объекты без дообучения (насколько хорошо — зависит от домена).

## 7.2 Архитектура

Три блока с очень разной стоимостью:

```
image ──► Image Encoder (тяжёлый, ~1 раз на картинку) ──► image embedding 64×64×256 ─┐
                                                                                     ├─► Mask Decoder (лёгкий) ──► 3 маски + IoU
prompts ──► Prompt Encoder (лёгкий) ─────────────────────► prompt tokens ─────────────┘
```

### Image Encoder

- **ViT-H/16**, предобученный как **MAE** (masked autoencoder); есть варианты ViT-L и ViT-B;
- вход **1024 × 1024** (длинная сторона ресайзится до 1024, остальное — padding) → патчи 16×16 → сетка **64 × 64**;
- для скорости большинство блоков — **windowed attention** (окна 14×14), и только 4 равномерно расставленных блока — глобальный attention;
- neck: $1\times1$ conv → $3\times3$ conv (по 256 каналов, с LayerNorm) → **image embedding $64 \times 64 \times 256$**.

Encoder запускается **один раз на картинку**, после чего на один embedding можно подавать сколько угодно подсказок.

### Prompt Encoder

**Sparse prompts** (точки, боксы) → токены размерности 256:

$$
e_{point} = PE(x, y) + e_{type}
$$

- $PE(x, y)$ — **фиксированный** (необучаемый) positional encoding на **случайных фурье-признаках** (Tancik et al., 2020): координаты умножаются на случайную гауссову матрицу, берутся $\sin$ и $\cos$;
- $e_{type}$ — **обучаемый** эмбеддинг типа: положительная точка, отрицательная точка, левый-верхний угол бокса, правый-нижний угол;
- бокс = **пара токенов** (два угла).

**Dense prompt** (маска):
- маска подаётся в разрешении в 4 раза меньше входа (256 × 256), две свёртки $2\times2$ со stride 2 + $1\times1$ conv → $64 \times 64 \times 256$;
- результат **поэлементно складывается** с image embedding; если маски нет — прибавляется обучаемый эмбеддинг «no mask».

### Mask Decoder

1. **Токены:** к prompt-токенам добавляются обучаемые **output tokens** (по одному на каждую выходную маску) и **IoU token**.
2. **Two-way transformer** (2 слоя). В каждом слое:
   - self-attention по токенам;
   - cross-attention **токены → изображение** (токены «читают» картинку);
   - MLP по токенам;
   - cross-attention **изображение → токены** (embedding картинки обновляется с учётом подсказки).
3. **Upsampling:** image embedding увеличивается в 4 раза двумя transposed conv: $64 \to 256$.
4. **Динамический классификатор:** каждый output token проходит через 3-слойный MLP → вектор $w_i$; маска — скалярное произведение в каждом пикселе:
$$
\hat{M}_i(x, y) = w_i^\top \, E_{up}(x, y)
$$
5. **IoU head:** MLP по IoU token предсказывает ожидаемый IoU каждой маски с «правильной» — это **score** для ранжирования.
6. Логиты $256 \times 256$ билинейно ресайзятся в исходный размер; бинарная маска: $\hat M_i > 0$ (то есть $\sigma(\hat M_i) > 0.5$).

Prompt encoder + mask decoder работают за **~50 мс** (в браузере на CPU) при готовом embedding → интерактивная разметка в реальном времени.

### Неоднозначность подсказок

Точка на рубашке человека может означать «рубашку», «человека» или «группу людей». Поэтому SAM для одной подсказки выдаёт **3 маски**
(условно: целое / часть / подчасть) + score каждой.

- При обучении считается loss для всех трёх, но градиент идёт только от **лучшей** (минимальный loss) — маски не «усредняются» в одну размытую;
- если подсказок несколько (неоднозначность меньше), используется отдельный 4-й output token → одна маска.

> Инсайд: На самом деле нутри модели всегда считаются 4 маски, а наружу отдаются либо 1, либо 3.
> Внутри: decoder всегда прогоняет все 4 output-токена и получает 4 маски (и 4 IoU-score). Наружу, в зависимости от флага:
> - multimask_output=True → отдаются маски 1, 2, 3 (три гипотезы);
> - multimask_output=False → отдаётся маска 0 (одна).


### Итого схема Mask Decoder
```
 [IoU, mask0, mask1, mask2, mask3, точка1, бокс_л, бокс_п]      image embedding 64×64×256
                                                                + dense prompt (маска / no_mask)
                         ↓                                              ↓
   ┌──────────────── Two-way блок ×2 ─────────────────────────────────────┐
   │  Self-Attention по токенам                                           │
   │  Cross-Attention: токены → картинка                                  │
   │  MLP по токенам                                                      │
   │  Cross-Attention: картинка → токены                                  │
   └──────────────────────────────────────────────────────────────────────┘
                         ↓
        финальный Cross-Attention: токены → картинка
                         ↓
       ┌─────────────────┼───────────────────────────┐
  image embedding    mask-токены (4)             IoU-токен
       ↓                 ↓                           ↓
 TransposedConv ×2   свой MLP на каждый токен      MLP
 64×64 → 256×256     → w0..w3 (по 32 числа)     → 4 IoU-score
 (32 канала)             ↓
    E_up(x,y)            ↓
       └──── скалярное произведение ────┘
                         ↓
              M0..M3 (логиты 256×256)
                         ↓
      multimask_output: False → M0,  True → M1, M2, M3
                         ↓
   ресайз до 1024 → обрезка padding → ресайз до H×W → порог: логит > 0
```


## 7.3 Обучение

### Loss

$$
\mathcal{L} = 20 \cdot \mathcal{L}_{focal} + 1 \cdot \mathcal{L}_{dice} \;\;(\text{по маске}) \; + \; \text{MSE}(\text{IoU}_{pred}, \text{IoU}_{true}) \;\;(\text{IoU head})
$$

### Имитация интерактивной разметки
SAM обучается быть интерактивным — человек кликает, модель улучшает маску. Чтобы это сымитировать при обучении.
11 раундов на каждую маску:
1. первая подсказка — случайная точка внутри объекта или бокс (с шумом);
2. 8 раундов: следующая точка сэмплируется из **области ошибки** текущего предсказания (FN → положительная, FP → отрицательная),
   а предыдущие логиты маски подаются как mask prompt;
3. ещё 2 раунда без новых точек — модель учится уточнять собственную маску.

### Data engine и SA-1B

Разметить миллиард масок вручную невозможно — данные собирались **вместе с моделью**, в три стадии:

1. **Assisted-manual:** разметчики кликают в браузерном инструменте на базе SAM, модель предлагает маски; SAM периодически переобучается на новых данных;
2. **Semi-automatic:** SAM сам находит уверенные маски, разметчики дорисовывают пропущенные объекты → разнообразие;
3. **Fully automatic:** сетка 32 × 32 точек-подсказок по картинке, фильтрация по предсказанному IoU и **стабильности** маски
   (маска почти не меняется при сдвиге порога), NMS.

Итог — **SA-1B: 11M изображений, 1.1B масок**. Почти все маски сгенерированы моделью автоматически на последней стадии.

Тот же алгоритм автоматической разметки доступен как `SamAutomaticMaskGenerator` — им пользуемся на семинаре.

### Zero-shot возможности и текст

- В статье SAM проверяют на 23 датасетах сегментации по одной точке, а также на задачах, которым его не учили: детекция границ, генерация object proposals, instance segmentation (боксы от детектора → SAM).
- **Text-to-mask (proof-of-concept):** при обучении в качестве подсказки подавали CLIP-эмбеддинг **изображения** кропа объекта, а на инференсе — CLIP-эмбеддинг **текста** (пространства CLIP согласованы).
  Работает, но авторы называют это исследовательским результатом.

### Почему SAM работает

1. Большой датасет
   SA-1B (11M изображений, 1.1B масок) обеспечивает огромную вариативность объектов.

2. Prompt-conditioned segmentation

    SAM обучается решать задачу
    
    $$
    M = f(I, prompt)
    $$
    
    а не классифицировать пиксели.

3. Универсальный decoder

    Mask decoder фактически работает как **conditional mask generator**, который может принимать различные типы сигналов.


### Ограничения SAM

- нет семантики: маска не имеет класса => для "найди всех коржей" нужна связка с детектором или текстовой моделью;
- может пропускать тонкие структуры и давать мелкие «оторванные» компоненты; границы не всегда такие чёткие, как у специализированных моделей;
- тяжёлый image encoder (ViT-H): интерактивность есть только при заранее посчитанном embedding;
- в специфичных доменах (медицина, спутники, микроскопия) zero-shot качество заметно падает => надо дообучть / использоватьт доменные версии.

## 7.4 SAM 2 (Ravi et al., 2024): картинки и видео

*«SAM 2: Segment Anything in Images and Videos»*

- задача обобщена на видео: подсказка на одном кадре → **masklet** (маска объекта во времени), подсказки можно добавлять на любых кадрах;
- **streaming memory:** кадры обрабатываются по одному; **memory encoder** кодирует предсказанные маски, **memory bank** хранит память
  последних кадров (FIFO, $N = 6$) + кадров с подсказками + «object pointers», **memory attention** (4 блока) обуславливает текущий кадр на память;
- для картинки память пустая — это просто SAM; image encoder — иерархический **Hiera** вместо ViT, поэтому на картинках SAM 2 **≈6× быстрее** SAM при сопоставимом качестве;
- датасет **SA-V:** 50.9K видео, 642.6K masklets, 35.5M масок (тоже через data engine).

## 7.5 SAM 3 (Carion et al., 2025): сегментация по концептам

*«SAM 3: Segment Anything with Concepts»*

- новая задача — **Promptable Concept Segmentation (PCS):** подсказка — **короткая фраза** («жёлтый школьный автобус») и/или **пример-изображение**,
  модель возвращает маски и id **всех** подходящих объектов на картинке/видео (а не одного объекта под точкой);
- детектор + трекер с памятью (как в SAM 2) на общем backbone, отдельная **presence head** — «есть ли концепт на картинке вообще»;
- данные — порядка 4M уникальных концептов, новый бенчмарк SA-Co.

Все три модели доступны в `transformers` (`SamModel`, `Sam2Model`, `Sam3Model`); у SAM 3 веса выдаются по запросу доступа на Hugging Face.


### Посмотрим на распределение параметров SAM ViT-B

In [ ]:
from segment_anything import sam_model_registry

sam = sam_model_registry["vit_b"](checkpoint=None) 
for name in ["image_encoder", "prompt_encoder", "mask_decoder"]:
    n = sum(p.numel() for p in getattr(sam, name).parameters())
    print(f"{name:>15}: {n / 1e6:7.2f}M")
print("positional encoding — обучаемый параметр?", any("positional_encoding_gaussian_matrix" in n for n, _ in sam.prompt_encoder.named_parameters()))
print("  ... или буфер:", [n for n, _ in sam.prompt_encoder.named_buffers()])

  image_encoder:   89.67M
 prompt_encoder:    0.01M
   mask_decoder:    4.06M
positional encoding — обучаемый параметр? False
  ... или буфер: ['pe_layer.positional_encoding_gaussian_matrix']
